# The mobile base

Reachy 2 is mounted on a mobile base!

> Make sure you have **enough clear space around the robot** before starting the tutorial. Required space is about 1.5m around the robot in each direction.

## Initialize your robot

First connect to your robot:

In [ ]:
from reachy2_sdk import ReachySDK
import time

reachy = ReachySDK(host='localhost')  # Replace with the actual IP

Let's check what contains the mobile base part:

In [ ]:
reachy.mobile_base

## Odometry

## Move around with gotos

The goto commands and goto-based commands described below follow all the rules you saw in the [goto introduction tutorial](2_goto_introduction.ipynb).

### Goto and odometry

The goto function is used to place the mobile_base at a relative position and orientation to its odometry, set when the robot is switched on. To be sure, you can reset the odometry before calling the function. 

In [ ]:
reachy.mobile_base.reset_odometry()

In [ ]:
reachy.mobile_base.odometry

Let's turn on the mobile to be able to move is around

In [ ]:
reachy.mobile_base.turn_on()

The robot is currently positionned at x=0, y=0, theta=0.  
If you want to move forward again the robot, you need to increase the x value (value is in meters):

In [ ]:
# Move 20 cm forward
a = reachy.mobile_base.goto(x=0.2, y=0.0, theta=0.0)

Now, request `goto(0, 0, 0)`. The robot will return to its previous position:

In [ ]:
reachy.mobile_base.goto(x=0.0, y=0.0, theta=0.0)

All the positions are relative to the fixed odometry coordinate system of the mobile base, set at the start of the robot of after a `reset_odometry()` asked by the user.

So if you do:

In [ ]:
# Move 30cm forward, to reach x=30cm
reachy.mobile_base.goto(x=0.3, y=0.0, theta=0.0)

# Go back by 10cm, to reach x=20cm
reachy.mobile_base.goto(x=0.2, y=0.0, theta=0.0)

The mobile is first going to the position x=30cm in the odometry frame. We then ask for it to get to the position x=20cm in this same frame, so the mobile base is going backward by 10cm to reach its new target.  

Let's do the same by resetting the odometry between the two commands:

In [ ]:
# Move 30cm forward, to reach x=30cm in the current odometry frame
reachy.mobile_base.goto(x=0.3, y=0.0, theta=0.0, wait=True)
time.sleep(0.5)
print(f"x position before odometry reset: {round(reachy.mobile_base.odometry['x'], 2)}")

# Reset odometry
reachy.mobile_base.reset_odometry()
time.sleep(0.5)
print(f"x position after odometry reset: {round(reachy.mobile_base.odometry['x'], 2)}")

# Move 20cm forward, to reach x=20cm in the new current odometry frame
reachy.mobile_base.goto(x=0.2, y=0.0, theta=0.0)

As we reset the odometry between the two commands, the mobile base odometry position is reset to x=0cm before the second command. It will then reach x=20cm in the new frame, so move forward by 20cm.

We recommend experimenting with this concept to get familiar.

### Relative moves

You can also decide to assign movements to the robot based on its current position and not on its odometry. 

Two methods are available to give relative orders:
- **`translate_by()`**: to give translations orders. 
- **`rotate_by()`**: to give rotations orders  

These methods work like all gotos, and return a GoToId.  

The translation or rotation is computed based on the current position if no goto is playing, or on the position required for the last queued or playing goto in case gotos are not finished.

Let's try some examples to better understand how it works.

#### translate_by()

Send the mobile base back the odometry frame origin first, and reset it:

In [ ]:
a = reachy.mobile_base.goto(x=0, y=0.0, theta=0.0, wait=True, timeout=10)
time.sleep(0.5)
reachy.mobile_base.reset_odometry()

Now, we are going to compare a `goto()` to a `translate_by()` command.  

If we check the odometry, we will see the mobile_base at the origin of the current frame (that we have just reset):

In [ ]:
time.sleep(0.5)
print("Odometry:")
print(f"'x': {round(reachy.mobile_base.odometry['x'], 2)}")
print(f"'y': {round(reachy.mobile_base.odometry['y'], 2)}")
print(f"'theta': {round(reachy.mobile_base.odometry['theta'], 2)}")

You can move forward to the position x=20cm by asking a translation of 20cm on the x-axis:

In [ ]:
reachy.mobile_base.translate_by(x = 0.2, y = 0.0)

Because you started from the origin, the result is the same as asking a `goto(x=0.2, y=0, theta=0)`.
If we send this goto:

In [ ]:
reachy.mobile_base.goto(x=0.2, y=0.0, theta=0.0, wait=True, timeout=10)

The mobile does not move, because we are already at this position. We can check this using the odometry:

In [ ]:
time.sleep(0.5)
print("Odometry:")
print(f"'x': {round(reachy.mobile_base.odometry['x'], 2)}")
print(f"'y': {round(reachy.mobile_base.odometry['y'], 2)}")
print(f"'theta': {round(reachy.mobile_base.odometry['theta'], 2)}")

But if we ask a new translation of 20cm, the mobile base will go forward:

In [ ]:
reachy.mobile_base.translate_by(x = 0.2, y = 0.0, wait=True, timeout=5)
time.sleep(0.5)

# Read odometry
print("Odometry:")
print(f"'x': {round(reachy.mobile_base.odometry['x'], 2)}")
print(f"'y': {round(reachy.mobile_base.odometry['y'], 2)}")
print(f"'theta': {round(reachy.mobile_base.odometry['theta'], 2)}")

#### rotate_by()

The `rotate_by()` method works quite the same way as the `translate_by()`, unlike it is for rotations.  
For example, you can go back to the initial position then rotate the mobile base. 

In [ ]:
# Go back to the initial position
reachy.mobile_base.goto(x=0.0, y=0.0, theta=0.0, wait=True)

# Rotation to be at 90 degrees in the frame
reachy.mobile_base.goto(x=0.0, y=0.0, theta=90.0, wait=True)

In [ ]:
time.sleep(0.5)
print("Odometry:")
print(f"'x': {round(reachy.mobile_base.odometry['x'], 2)}")
print(f"'y': {round(reachy.mobile_base.odometry['y'], 2)}")
print(f"'theta': {round(reachy.mobile_base.odometry['theta'], 2)}")

Now, the mobile base can be rotated 90° from its current position, allowing to get a odometry with a theta = 0°. 

In [ ]:
reachy.mobile_base.rotate_by(theta=-90.0, wait=True)
time.sleep(0.5)

# Check the odometry
print("Odometry:")
print(f"'x': {round(reachy.mobile_base.odometry['x'], 2)}")
print(f"'y': {round(reachy.mobile_base.odometry['y'], 2)}")
print(f"'theta': {round(reachy.mobile_base.odometry['theta'], 2)}")

#### Choose the right method!

Be careful with the method you use. For example, those two sequences have completely different results:

In [ ]:
reachy.mobile_base.goto(x=0, y=0.0, theta=0.0, timeout=10)
reachy.mobile_base.goto(x=0.2, y=0.0, theta=0.0, timeout=10)
reachy.mobile_base.goto(x=0, y=0.4, theta=0.0, timeout=10)
reachy.mobile_base.goto(x=0, y=0, theta=90.0, timeout=10, wait=True)

In [ ]:
time.sleep(0.5)
print("Odometry:")
print(f"'x': {round(reachy.mobile_base.odometry['x'], 2)}")
print(f"'y': {round(reachy.mobile_base.odometry['y'], 2)}")
print(f"'theta': {round(reachy.mobile_base.odometry['theta'], 2)}")

In [ ]:
reachy.mobile_base.goto(x=0, y=0.0, theta=0.0, timeout=10)
reachy.mobile_base.translate_by(x=0.2, y=0.0, timeout=10)
reachy.mobile_base.translate_by(x=0, y=0.4, timeout=10)
a=reachy.mobile_base.rotate_by(theta=90.0, timeout=10, wait=True)

In [ ]:
time.sleep(0.5)
print("Odometry:")
print(f"'x': {round(reachy.mobile_base.odometry['x'], 2)}")
print(f"'y': {round(reachy.mobile_base.odometry['y'], 2)}")
print(f"'theta': {round(reachy.mobile_base.odometry['theta'], 2)}")

### Goto tolerances

Unlike the arms and head whose movements duration is based on a duration argument, a mobile base goto duration is based on the tolerances and timeout arguments you can choose.  

You can modify two different tolerances:
- the **distance_tolerance**: defines the maximum distance to the (x, y) position in meters to consider the goto as finished (even if movement to target is not over yet)
- the **angle_tolerance**: defines the angle distance to the theta rotation to consider the goto as finished (even if movement to target is not over yet) *- units based on the degrees argument, in degrees by default*

> Default distance_tolerance is **0.05 meter**  
> Default angle_tolerance is **5 degrees**

Be within those tolerances will raise the flag that the goto is considered as finished, which **does not mean the movement is over**! You can start a new goto without finishing this one, but if no other goto has been sent, the mobile base will try to get as close as possible to the target.  
As it is easier to understand with examples, let's take some:

In [ ]:
# Going back to the base position
reachy.mobile_base.goto(x=0, y=0.0, theta=0.0)

We are first going to send a target to x=0.6m, with a tolerance of 20cm. 
Put the `wait` argument to True to wait for the *goto* to be finished:

In [ ]:
reachy.mobile_base.goto(x=0.6, y=0.0, theta=0.0, distance_tolerance=0.2, wait=True)

odom_goto_end = reachy.mobile_base.odometry
print("Odometry after the goto is declared as finished:")
print(f"'x': {round(odom_goto_end['x'], 3)}")
print(f"'y': {round(odom_goto_end['y'], 3)}")
print(f"'theta': {round(odom_goto_end['theta'], 3)}")

time.sleep(1)
odom_move_end = reachy.mobile_base.odometry
print("Odometry after the movement finished:")
print(f"'x': {round(odom_move_end['x'], 3)}")
print(f"'y': {round(odom_move_end['y'], 3)}")
print(f"'theta': {round(odom_move_end['theta'], 3)}")

When the goto was **declared as finished**, to mobile base was **still receiving commands** to try to get as close to the target position as it could, so the **movement was not finished**. Simply, you can see the odometry values are within the tolerances, so the robot is ready to receive a new goto goal.  

This can be used when chaining gotos.  

For example, if we do this chain of gotos with the default tolerances:

In [ ]:
goto1 = reachy.mobile_base.goto(x=0.0, y=0.0, theta=0.0)
goto2 = reachy.mobile_base.goto(x=0.6, y=0.0, theta=0.0)
goto3 = reachy.mobile_base.goto(x=-0.2, y=0.0, theta=0.0)
goto4 = reachy.mobile_base.goto(x=0.3, y=0.0, theta=0.0)

while not reachy.is_goto_finished(goto1):
    time.sleep(0.05)
odom1 = reachy.mobile_base.odometry
print("Odometry after goto1:")
print(f"'x': {round(odom1['x'], 3)}")
print(f"'y': {round(odom1['y'], 3)}")
print(f"'theta': {round(odom1['theta'], 3)}")

while not reachy.is_goto_finished(goto2):
    time.sleep(0.05)
odom2 = reachy.mobile_base.odometry
print("Odometry after goto2:")
print(f"'x': {round(odom2['x'], 3)}")
print(f"'y': {round(odom2['y'], 3)}")
print(f"'theta': {round(odom2['theta'], 3)}")

while not reachy.is_goto_finished(goto3):
    time.sleep(0.05)
odom3 = reachy.mobile_base.odometry
print("Odometry after goto3:")
print(f"'x': {round(odom3['x'], 3)}")
print(f"'y': {round(odom3['y'], 3)}")
print(f"'theta': {round(odom3['theta'], 3)}")

while not reachy.is_goto_finished(goto4):
    time.sleep(0.05)
odom4 = reachy.mobile_base.odometry
print("Odometry after goto4:")
print(f"'x': {round(odom4['x'], 3)}")
print(f"'y': {round(odom4['y'], 3)}")
print(f"'theta': {round(odom4['theta'], 3)}")

We wait each time for the current goto to be finished before we read the odometry. As we see, the odometry is close to the target after the goto is declared as finished, as the default distance_tolerance is 5cm.  

If we do now the same with the higher tolerances:

In [ ]:
goto1 = reachy.mobile_base.goto(x=0.0, y=0.0, theta=0.0, distance_tolerance=0.1)
goto2 = reachy.mobile_base.goto(x=0.6, y=0.0, theta=0.0, distance_tolerance=0.3)
goto3 = reachy.mobile_base.goto(x=-0.2, y=0.0, theta=0.0, distance_tolerance=0.3)
goto4 = reachy.mobile_base.goto(x=0.3, y=0.0, theta=0.0, distance_tolerance=0.1)

while not reachy.is_goto_finished(goto1):
    time.sleep(0.05)
odom1 = reachy.mobile_base.odometry
print("Odometry after goto1:")
print(f"'x': {round(odom1['x'], 3)}")
print(f"'y': {round(odom1['y'], 3)}")
print(f"'theta': {round(odom1['theta'], 3)}")

while not reachy.is_goto_finished(goto2):
    time.sleep(0.05)
odom2 = reachy.mobile_base.odometry
print("Odometry after goto2:")
print(f"'x': {round(odom2['x'], 3)}")
print(f"'y': {round(odom2['y'], 3)}")
print(f"'theta': {round(odom2['theta'], 3)}")

while not reachy.is_goto_finished(goto3):
    time.sleep(0.05)
odom3 = reachy.mobile_base.odometry
print("Odometry after goto3:")
print(f"'x': {round(odom3['x'], 3)}")
print(f"'y': {round(odom3['y'], 3)}")
print(f"'theta': {round(odom3['theta'], 3)}")

while not reachy.is_goto_finished(goto4):
    time.sleep(0.05)
odom4 = reachy.mobile_base.odometry
print("Odometry after goto4:")
print(f"'x': {round(odom4['x'], 3)}")
print(f"'y': {round(odom4['y'], 3)}")
print(f"'theta': {round(odom4['theta'], 3)}")

We still wait for the gotos to be finished The next goto starts as soon as the tolerance of the current one is reached, so the mobile base is further to the target when switching to the next command.  

When there is no more queued goto, the mobile base tries to reach the target position, even if the *goto* was declared finished. If we ask again for the odometry, without asking any new goto, the robo has finally reached its target:

In [ ]:
time.sleep(1)
odom4_bis = reachy.mobile_base.odometry
print("Odometry after goto4:")
print(f"'x': {round(odom4_bis['x'], 3)}")
print(f"'y': {round(odom4_bis['y'], 3)}")
print(f"'theta': {round(odom4_bis['theta'], 3)}")

> Note : setting the tolerances to 0 will make the target impossible to reach in most cases. You will have to wait for the goto timeout to be over before the mobile base will start another goto.

### Goto timeout

The timeout will stop the movement whatever the current position is if the target has not been reached before the timeout time is over.  
In the case the target has been reached, the timeout has no effect.  

Contrary to the tolerances, a **goto finished by a timeout does also end the real movement on the robot**, which no more commands will be sent after the timeout and the robot will stop.  


To see how it works, let's send the mobile base back to its base position and then send a goto with a very low timeout:

In [ ]:
# Go back to position
reachy.mobile_base.goto(x=0.0, y=0.0, theta=0.0, wait=True)

# Use a low timeout
reachy.mobile_base.goto(x=0.4, y=0.0, theta=90.0, timeout=0.5)


If we check the odometry now:

In [ ]:
time.sleep(0.5)
print("Odometry:")
print(f"'x': {round(reachy.mobile_base.odometry['x'], 5)}")
print(f"'y': {round(reachy.mobile_base.odometry['y'], 2)}")
print(f"'theta': {round(reachy.mobile_base.odometry['theta'], 2)}")

The distance to the tolerance is still not good, but the movement has stopped after the given timeout of 5 seconds.

> Default timeout is **100 seconds**

You can also use the timeout to interrupt a goto after a certain amount of time in a given direction. For example:

In [ ]:
# Go back to base position
reachy.mobile_base.goto(x=0, y=0.0, theta=0.0, wait=True)

# Start movement torwards x=1m
timeout_goto = reachy.mobile_base.goto(x=1, y=0.0, theta=0.0, timeout=2)
tic = time.time()
while not reachy.is_goto_finished(timeout_goto):
    time.sleep(0.05)
print(f"goto interrupted after {round(time.time() - tic, 2)} seconds")

Be careful that the mobile does not precisely follow the expected trajectory to reach the target position.

## Advanced usage

### Set speed

The speed of the movement can be defined using the command `set_goal_speed()`. It requires to set all 3 target speeds *vx*, *vy* and *vtheta*.
Send then the command to the robot with `send_speed_command()`.

> **This will assign speed to the robot for 200ms**

In [ ]:
reachy.mobile_base.set_goal_speed(vx=0.5, vy=0.0, vtheta=0)
tic=time.time()
while time.time()-tic < 2:
    reachy.mobile_base.send_speed_command()
    time.sleep(0.01)

Setting speed can be useful when replaying movements for example.

## Free wheel

In order to be able to move easily the mobile base manually (pushing the robot for example), you can turn_off the mobile like other parts:

In [ ]:
reachy.mobile_base.turn_off()